# 02 - Spatial and temporal evaluation

**SCAFFOLD - NOT RUN.** Evaluation requires a versioned model, real timestamped inventory, and held-out spatial/temporal blocks.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%pip install -q numpy pandas scikit-learn

import random
import numpy as np
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

In [ ]:
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score, confusion_matrix

# Load predictions created for a strictly held-out spatial and later temporal test set.
predictions = pd.read_csv('/content/drive/MyDrive/terrasafe/held_out_predictions.csv')
truth = predictions['landslide_label'].to_numpy()
probability = predictions['probability'].to_numpy()
predicted = probability >= 0.5
tn, fp, fn, tp = confusion_matrix(truth, predicted, labels=[0, 1]).ravel()
metrics = {
    'accuracy': accuracy_score(truth, predicted),
    'precision': precision_score(truth, predicted, zero_division=0),
    'recall': recall_score(truth, predicted, zero_division=0),
    'f1': f1_score(truth, predicted, zero_division=0),
    'roc_auc': roc_auc_score(truth, probability),
    'pr_auc': average_precision_score(truth, probability),
    'confusion_matrix': [[int(tn), int(fp)], [int(fn), int(tp)]],
    'false_alarm_rate': fp / (fp + tn) if fp + tn else None,
    'miss_rate': fn / (fn + tp) if fn + tp else None,
}
metrics

In [ ]:
# Lead time is computable only when event onset and the first threshold-crossing prediction are timestamped.
required = {'event_onset_utc', 'first_threshold_crossing_utc'}
if required.issubset(predictions.columns):
    predictions['lead_time_hours'] = (pd.to_datetime(predictions['event_onset_utc'], utc=True) - pd.to_datetime(predictions['first_threshold_crossing_utc'], utc=True)).dt.total_seconds() / 3600
else:
    print('Lead-time check unavailable: required event and prediction timestamps are missing.')

This notebook has not been run. Do not report metrics until the held-out dataset and labels have been independently reviewed. Synthetic labels cannot validate operational performance.